# Othello Mamba-JEPA Training

Change only `CONFIG_NAME` to run one of the Mamba-JEPA YAML experiments.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Project Setup

In [3]:
from pathlib import Path
import os
# '1' replaces nested tqdm bars with the trainer's single-line chunk progress
# bar: one in-place updating line per chunk, then the permanent metrics line.
os.environ['OTHELLO_DISABLE_TQDM'] = '1'
import subprocess
import sys

PROJECT_ROOT = Path('/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/Master_Thesis_Artifacts')
DATA_ROOT = ARTIFACT_ROOT / 'data'
RUNS_ROOT = ARTIFACT_ROOT / 'runs' / 'Mamba-JEPA'
CONFIGS_DIR = PROJECT_ROOT / 'configs'

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert (PROJECT_ROOT / 'pyproject.toml').exists(), PROJECT_ROOT
assert ARTIFACT_ROOT.exists(), ARTIFACT_ROOT
assert DATA_ROOT.exists(), DATA_ROOT
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RUNS_ROOT   :', RUNS_ROOT)
print('CONFIGS_DIR :', CONFIGS_DIR)


PROJECT_ROOT: /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code
RUNS_ROOT   : /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-JEPA
CONFIGS_DIR : /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code/configs


## 3. Mamba Backend Check

In [4]:
import importlib.util
import subprocess
import sys
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

if importlib.util.find_spec('mamba_ssm') is None:
    print('mamba_ssm not found; installing official Mamba backend...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'packaging', 'ninja'], check=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'mamba-ssm[causal-conv1d]', '--no-build-isolation'],
        check=True,
    )
else:
    print('mamba_ssm already installed.')

import mamba_ssm
print('mamba_ssm OK:', getattr(mamba_ssm, '__version__', 'unknown'))


torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
mamba_ssm not found; installing official Mamba backend...
mamba_ssm OK: 2.3.2.post1


## 4. Select Config

In [5]:
import yaml

# Change this filename to run another Mamba-JEPA experiment.
# CONFIG_NAME = 'mamba_jepa_v1_hd_vicreg.yml'
# CONFIG_NAME = 'mamba_jepa_v5_hd_infonce.yml'
# CONFIG_NAME = 'mamba_jepa_v1_hd_vicreg_b8_all_positions.yml'
# CONFIG_NAME = 'mamba_jepa_v1_hd_vicreg_ema_b8_all_positions.yml'
CONFIG_NAME = 'mamba_jepa_v5_hd_infonce_b8_all_positions.yml'

CONFIG_PATH = CONFIGS_DIR / CONFIG_NAME
assert CONFIG_PATH.exists(), CONFIG_PATH

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    EXPERIMENT_CONFIG = yaml.safe_load(f)

RUN_NAME = EXPERIMENT_CONFIG['experiment']['name']
BOARD_SIZE = int(EXPERIMENT_CONFIG['data']['board_size'])
N_TRAIN_CHUNKS = int(EXPERIMENT_CONFIG['data'].get('li_split_train_chunks', 200))
DATA_DIR = DATA_ROOT / f'othello_{BOARD_SIZE}x{BOARD_SIZE}'
LOCAL_OUT_DIR = Path('/content/runs/Mamba-JEPA') / RUN_NAME
DRIVE_OUT_DIR = RUNS_ROOT / RUN_NAME

LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print('CONFIG_NAME   :', CONFIG_NAME)
print('RUN_NAME      :', RUN_NAME)
print('BOARD_SIZE    :', BOARD_SIZE)
print('N_TRAIN_CHUNKS:', N_TRAIN_CHUNKS)
print('DATA_DIR      :', DATA_DIR)
print('LOCAL_OUT_DIR :', LOCAL_OUT_DIR)
print('DRIVE_OUT_DIR :', DRIVE_OUT_DIR)
assert DATA_DIR.exists(), DATA_DIR


CONFIG_NAME   : mamba_jepa_v5_hd_infonce_b8_all_positions.yml
RUN_NAME      : mamba_jepa_v5_hd_infonce_allpos_b8
BOARD_SIZE    : 8
N_TRAIN_CHUNKS: 200
DATA_DIR      : /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_8x8
LOCAL_OUT_DIR : /content/runs/Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b8
DRIVE_OUT_DIR : /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b8


## 5. Optional Smoke Run

In [6]:
# Keep this enabled for a quick GPU end-to-end check before the full run.
RUN_SMOKE = True

if RUN_SMOKE:
    import importlib
    import torch
    import othello_research.training.train_mamba_jepa as tjm
    import othello_research.models.jepa_backbone as jepa_backbone
    import othello_research.models.mamba_ar as mamba_ar
    import othello_research.objectives.jepa as jepa_obj
    import othello_research.objectives.jepa_hard_disjoint_infonce as jepa_v5_obj
    import othello_research.objectives.jepa_hard_disjoint_action as jepa_v6_obj

    for module in [mamba_ar, jepa_backbone, jepa_obj, jepa_v5_obj, jepa_v6_obj, tjm]:
        importlib.reload(module)

    parser = tjm.build_parser()
    smoke_cli = [
        *tjm.config_to_cli_args(EXPERIMENT_CONFIG, parser),
        '--out_dir', str(Path('/content/mamba_jepa_smoke') / RUN_NAME),
        '--data_dir', str(DATA_DIR),
        '--batch_size', '4',
        '--branch_groups_per_batch', '2',
        '--branch_samples_per_group', '2',
        '--probe_interval_steps', '0',
    ]
    smoke_args = parser.parse_args(smoke_cli)
    smoke_cfg, _ = tjm.namespace_to_train_config(smoke_args)
    model, objective_cfg = tjm.build_objective_model(smoke_cfg)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.train()

    vocab_actions = BOARD_SIZE * BOARD_SIZE - 4
    context_len = 6
    position_pairs = None
    if tjm.objective_class(smoke_cfg) == 'jepa' and smoke_cfg.position_sampling == 'all':
        x = torch.randint(0, vocab_actions, (4, context_len + 1), device=device)
        out, position_pairs = tjm.forward_all_position_v1(
            model,
            x,
            pad_token=vocab_actions,
            target_chunk_size=smoke_cfg.all_position_target_chunk_size,
            stats_chunk_size=smoke_cfg.all_position_stats_chunk_size,
        )
    elif tjm.objective_class(smoke_cfg) == 'jepa':
        x_context = torch.randint(0, vocab_actions, (4, context_len), device=device)
        x_target = torch.randint(0, vocab_actions, (4, 1), device=device)
        target_positions = torch.full((4, 1), context_len, device=device, dtype=torch.long)
        out = model(x_context, x_target, target_positions=target_positions)
    else:
        x_context = torch.randint(0, vocab_actions, (4, context_len), device=device)
        x_context[:2] = x_context[0]
        x_context[2:] = x_context[2]
        prefix_ids = torch.tensor([0, 0, 1, 1], device=device)
        next_actions = torch.randint(0, vocab_actions, (4,), device=device)
        out = model(x_context, next_actions, prefix_ids)

    loss = out['loss']
    if not torch.isfinite(loss):
        raise RuntimeError(f'Smoke loss is not finite: {loss}')
    loss.backward()
    print('Smoke OK')
    print('device:', device)
    print('objective:', tjm.objective_class(smoke_cfg))
    print('position sampling:', smoke_cfg.position_sampling)
    if position_pairs is not None:
        print('all-position pairs:', position_pairs)
    print('loss:', float(loss.detach().cpu()))
    print('context params:', f'{sum(p.numel() for p in model.context_encoder.parameters()):,}')
else:
    print('Smoke run skipped.')


mamba length-1 fast path parity: PASS (max_abs=6.985e-08)
Smoke OK
device: cuda
objective: jepa
position sampling: all
all-position pairs: 24
loss: 1.501717448234558
context params: 25,529,856


## 6. Full Training

In [7]:
RESUME_FROM = None
# RESUME_FROM = str(LOCAL_OUT_DIR / 'latest.pt')

command = [
    sys.executable, '-u', '-m', 'othello_research.training.train_mamba_jepa',
    '--config', str(CONFIG_PATH),
    '--out_dir', str(LOCAL_OUT_DIR),
    '--data_dir', str(DATA_DIR),
    '--drive_sync_dir', str(DRIVE_OUT_DIR),
]
if RESUME_FROM:
    command.extend(['--resume', RESUME_FROM])

# Colab does not display child-process output written to inherited file
# descriptors, so pipe it and forward manually. Raw (non line-buffered)
# forwarding keeps the trainer's '\r'-refreshed chunk progress bar live.
import codecs
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
print('Running:', ' '.join(command), flush=True)
process = subprocess.Popen(
    command,
    cwd=PROJECT_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
assert process.stdout is not None
decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
while True:
    data = process.stdout.read1(8192)
    if not data:
        break
    sys.stdout.write(decoder.decode(data))
    sys.stdout.flush()
tail = decoder.decode(b'', True)
if tail:
    sys.stdout.write(tail)
    sys.stdout.flush()
returncode = process.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)
print('Training finished, returncode:', returncode)


Running: /usr/bin/python3 -u -m othello_research.training.train_mamba_jepa --config /content/drive/Othercomputers/MyLaptop/Master_Thesis_Code/configs/mamba_jepa_v5_hd_infonce_b8_all_positions.yml --out_dir /content/runs/Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b8 --data_dir /content/drive/MyDrive/Master_Thesis_Artifacts/data/othello_8x8 --drive_sync_dir /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b8
estimated total_steps=78125 (manifest, ~19,999,840 games over 1 pass(es))
device: cuda
train chunks: 200
val chunks: 38
vocab_size: 61, block_size: 59
jepa: objective_class=jepa variant=v1 view_mode=hard_disjoint_future loss=infonce predictor=linear horizon=1 position_sampling=all ema=0.996 precision=bf16
milestone games: [1000000, 5000000, 10000000, 15000000, 20000000]
fraction checkpoints: [0.1, 0.25, 0.5]
drive auto-sync: every 5 chunk(s) -> /content/drive/MyDrive/Master_Thesis_Artifacts/runs/Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_

## 7. Optional Disconnect

In [ ]:
from google.colab import runtime
runtime.unassign()
